# VI. Inject Realism Into Campaign Data

This notebook transforms the cleaned marketing dataset into a more realistic version with channel effects, audience-campaign interactions, seasonality, and controlled noise artifacts.

Output file: `../datasets/realistic_campaign_dataset.csv`

## Step 1: Imports and Paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

cleaned_dataset_path = Path("../datasets/marketing_campaign_dataset_cleaned.csv")
realistic_dataset_path = Path("../datasets/realistic_campaign_dataset.csv")

if not cleaned_dataset_path.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found at: {cleaned_dataset_path.resolve()}\n"
        "Run 02_clean_and_validate.ipynb first."
    )

realistic_dataset_path.parent.mkdir(parents=True, exist_ok=True)

print(f"Input:  {cleaned_dataset_path.resolve()}")
print(f"Output: {realistic_dataset_path.resolve()}")
print(f"Seed:   {SEED}")

Input:  C:\Users\hecto\OneDrive\Documentos\VSCode\py_marketing_eda\datasets\marketing_campaign_dataset_cleaned.csv
Output: C:\Users\hecto\OneDrive\Documentos\VSCode\py_marketing_eda\datasets\realistic_campaign_dataset.csv
Seed:   42


## Step 2: Load Cleaned Dataset

In [2]:
df = pd.read_csv(cleaned_dataset_path, parse_dates=["Date"])

required_columns = [
    "Date", "Channel_Used", "Campaign_Type", "Target_Audience",
    "Conversion_Rate", "Acquisition_Cost", "ROI",
    "Clicks", "Impressions", "Engagement_Score"
]
missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"Loaded shape: {df.shape}")
df.head()

Loaded shape: (200000, 16)


,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date
0,1,Innovate Industries,Email,Men 18-24,30,Google Ads,0.04,16174.0,6.29,Chicago,Spanish,506,1922,6,Health & Wellness,2021-01-01
1,2,NexGen Systems,Email,Women 35-44,60,Google Ads,0.12,11566.0,5.61,New York,German,116,7523,7,Fashionistas,2021-01-02
2,3,Alpha Innovations,Influencer,Men 25-34,30,YouTube,0.07,10200.0,7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,2021-01-03
3,4,DataTech Solutions,Display,All Ages,60,YouTube,0.11,12724.0,5.55,Miami,Mandarin,217,1820,7,Health & Wellness,2021-01-04
4,5,NexGen Systems,Email,Men 25-34,15,YouTube,0.05,16452.0,6.50,Los Angeles,Mandarin,379,4201,3,Health & Wellness,2021-01-05


## Step 3: Apply Realism Transformations

This cell injects:
- channel efficiency differences
- audience-campaign-channel interactions
- seasonality and event effects
- funnel-consistent metric updates
- saturation and noise artifacts

In [3]:
realistic_df = df.copy()

# Keep numeric fields stable before transformations.
realistic_df["Conversion_Rate"] = pd.to_numeric(realistic_df["Conversion_Rate"], errors="coerce").fillna(0.05)
realistic_df["Acquisition_Cost"] = pd.to_numeric(realistic_df["Acquisition_Cost"], errors="coerce").fillna(10000.0)
realistic_df["ROI"] = pd.to_numeric(realistic_df["ROI"], errors="coerce").fillna(2.0)
realistic_df["Clicks"] = pd.to_numeric(realistic_df["Clicks"], errors="coerce").fillna(100).clip(lower=1)
realistic_df["Impressions"] = pd.to_numeric(realistic_df["Impressions"], errors="coerce").fillna(1000).clip(lower=10)
realistic_df["Engagement_Score"] = pd.to_numeric(realistic_df["Engagement_Score"], errors="coerce").fillna(5.0)

# Calendar components used for seasonality and event windows.
date_series = pd.to_datetime(realistic_df["Date"], errors="coerce")
month = date_series.dt.month.fillna(6).astype(int)
day_of_week = date_series.dt.dayofweek.fillna(3).astype(int)
day_of_year = date_series.dt.dayofyear.fillna(180).astype(int)

monthly_wave = 0.12 * np.sin(2 * np.pi * month / 12)
weekly_wave = 0.05 * np.cos(2 * np.pi * day_of_week / 7)
trend = np.linspace(-0.03, 0.05, len(realistic_df))
is_peak_month = month.isin([11, 12]).astype(float)
event_wave = 0.08 * is_peak_month
season_factor = (1 + monthly_wave + weekly_wave + trend + event_wave).clip(0.75, 1.40)

# Channel-level effects make efficiency visibly different across channels.
channel_cost_multiplier = {
    "Google Ads": 0.92, "YouTube": 1.12, "Instagram": 0.95,
    "Facebook": 1.00, "Email": 0.75, "Website": 0.88
}
channel_ctr_multiplier = {
    "Google Ads": 1.15, "YouTube": 0.78, "Instagram": 0.98,
    "Facebook": 0.93, "Email": 0.82, "Website": 1.05
}
channel_conv_multiplier = {
    "Google Ads": 1.18, "YouTube": 0.82, "Instagram": 1.02,
    "Facebook": 0.95, "Email": 0.90, "Website": 1.08
}
channel_reach_multiplier = {
    "Google Ads": 0.95, "YouTube": 1.25, "Instagram": 1.08,
    "Facebook": 1.02, "Email": 0.72, "Website": 0.85
}

channel = realistic_df["Channel_Used"].astype(str)
campaign = realistic_df["Campaign_Type"].astype(str)
audience = realistic_df["Target_Audience"].astype(str)

ch_cost = channel.map(channel_cost_multiplier).fillna(1.0).to_numpy()
ch_ctr = channel.map(channel_ctr_multiplier).fillna(1.0).to_numpy()
ch_conv = channel.map(channel_conv_multiplier).fillna(1.0).to_numpy()
ch_reach = channel.map(channel_reach_multiplier).fillna(1.0).to_numpy()

# Audience and campaign interaction lifts.
audience_lift = {
    "Men 18-24": 1.10, "Men 25-34": 1.06, "Women 25-34": 1.08,
    "Women 35-44": 0.98, "All Ages": 0.93
}
campaign_lift = {
    "Search": 1.16, "Email": 0.96, "Social Media": 1.03,
    "Influencer": 0.94, "Display": 0.90
}

aud = audience.map(audience_lift).fillna(1.0).to_numpy()
camp = campaign.map(campaign_lift).fillna(1.0).to_numpy()

# Extra interaction signal: young audiences on social/video perform better.
is_young = audience.isin(["Men 18-24", "Women 25-34"]).to_numpy()
is_social_video = channel.isin(["Instagram", "YouTube", "Facebook"]).to_numpy()
interaction_lift = np.where(is_young & is_social_video, 1.12, 0.98)

# Build realistic impressions first with seasonality and channel reach.
base_impressions = realistic_df["Impressions"].to_numpy(dtype=float)
impression_noise = rng.lognormal(mean=0.0, sigma=0.15, size=len(realistic_df))
new_impressions = np.round(base_impressions * ch_reach * season_factor.to_numpy() * impression_noise).astype(int)
new_impressions = np.clip(new_impressions, 50, None)

# CTR and clicks with saturation at very high reach.
old_ctr = (realistic_df["Clicks"].to_numpy(dtype=float) / np.clip(base_impressions, 1, None))
old_ctr = np.clip(old_ctr, 0.002, 0.45)
saturation = 1 / (1 + (new_impressions / np.percentile(new_impressions, 80)) ** 1.2)
ctr_noise = rng.normal(loc=1.0, scale=0.08, size=len(realistic_df))
new_ctr = old_ctr * ch_ctr * aud * season_factor.to_numpy() * saturation * ctr_noise
new_ctr = np.clip(new_ctr, 0.001, 0.40)
new_clicks = np.round(new_impressions * new_ctr).astype(int)
new_clicks = np.clip(new_clicks, 1, None)

# Conversion rate combines channel, audience, campaign, and interaction effects.
base_cvr = realistic_df["Conversion_Rate"].to_numpy(dtype=float)
cvr_noise = rng.normal(loc=1.0, scale=0.10, size=len(realistic_df))
new_cvr = base_cvr * ch_conv * aud * camp * interaction_lift * season_factor.to_numpy() * cvr_noise
new_cvr = np.clip(new_cvr, 0.003, 0.45)

# Acquisition cost varies by channel and seasonal pressure.
base_cost = realistic_df["Acquisition_Cost"].to_numpy(dtype=float)
cost_noise = rng.lognormal(mean=0.0, sigma=0.18, size=len(realistic_df))
new_acq_cost = base_cost * ch_cost * (1 + 0.2 * (1 - saturation)) * (1 + 0.3 * is_peak_month.to_numpy()) * cost_noise
new_acq_cost = np.clip(new_acq_cost, 50, None)

# Build spend/revenue/ROI from funnel mechanics.
spend = new_clicks * new_acq_cost
old_spend = realistic_df["Clicks"].to_numpy(dtype=float) * realistic_df["Acquisition_Cost"].to_numpy(dtype=float)
old_revenue = old_spend * (1 + realistic_df["ROI"].to_numpy(dtype=float))
base_value_per_conversion = old_revenue / np.clip(realistic_df["Clicks"].to_numpy(dtype=float) * realistic_df["Conversion_Rate"].to_numpy(dtype=float), 1, None)
base_value_per_conversion = np.clip(base_value_per_conversion, 20, np.percentile(base_value_per_conversion, 99))

conversions = new_clicks * new_cvr
value_noise = rng.normal(loc=1.0, scale=0.12, size=len(realistic_df))
campaign_value_lift = np.where(campaign == "Search", 1.08, np.where(campaign == "Display", 0.92, 1.00))
revenue = conversions * base_value_per_conversion * campaign_value_lift * value_noise
new_roi = (revenue - spend) / np.clip(spend, 1, None)

# Inject rare anomalies to mimic tracking/reporting artifacts.
anomaly_idx = rng.choice(len(realistic_df), size=max(1, int(0.005 * len(realistic_df))), replace=False)
new_roi[anomaly_idx] = new_roi[anomaly_idx] * rng.uniform(0.25, 1.8, size=len(anomaly_idx))

# Write transformed columns back.
realistic_df["Impressions"] = new_impressions.astype(int)
realistic_df["Clicks"] = new_clicks.astype(int)
realistic_df["Conversion_Rate"] = np.round(new_cvr, 4)
realistic_df["Acquisition_Cost"] = np.round(new_acq_cost, 2)
realistic_df["ROI"] = np.round(new_roi, 4)

# Slightly reshape engagement with channel and season influence.
engagement_noise = rng.normal(loc=0.0, scale=1.15, size=len(realistic_df))
new_engagement = realistic_df["Engagement_Score"].to_numpy(dtype=float) + (2.0 * (ch_reach - 1.0)) + (3.0 * monthly_wave.to_numpy()) + engagement_noise
realistic_df["Engagement_Score"] = np.clip(np.round(new_engagement), 1, 10).astype(int)

# Add a small realistic missingness pattern in engagement only.
missing_mask = (channel == "Display") & (day_of_week == 6) & (rng.random(len(realistic_df)) < 0.12)
realistic_df.loc[missing_mask, "Engagement_Score"] = np.nan

realistic_df.head()

,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date
0,1,Innovate Industries,Email,Men 18-24,30,Google Ads,0.0568,14316.89,9.1107,Chicago,Spanish,638,1883,6.0,Health & Wellness,2021-01-01
1,2,NexGen Systems,Email,Women 35-44,60,Google Ads,0.1270,11942.21,5.3323,New York,German,63,6230,7.0,Fashionistas,2021-01-02
2,3,Alpha Innovations,Influencer,Men 25-34,30,YouTube,0.0510,13378.11,3.4359,Los Angeles,French,294,11428,1.0,Outdoor Adventurers,2021-01-03
3,4,DataTech Solutions,Display,All Ages,60,YouTube,0.0924,18794.08,2.9964,Miami,Mandarin,188,2829,5.0,Health & Wellness,2021-01-04
4,5,NexGen Systems,Email,Men 25-34,15,YouTube,0.0448,25978.28,3.7850,Los Angeles,Mandarin,223,4159,3.0,Health & Wellness,2021-01-05


## Step 4: Save Realistic Dataset

In [4]:
realistic_df.to_csv(realistic_dataset_path, index=False)

print(f"Saved realistic dataset to: {realistic_dataset_path.resolve()}")
print(f"Rows: {len(realistic_df)}")

comparison = pd.DataFrame({
    "metric": [
        "roi_std", "roi_mean", "conversion_rate_std",
        "acquisition_cost_std", "clicks_impressions_corr"
    ],
    "cleaned": [
        df["ROI"].std(),
        df["ROI"].mean(),
        df["Conversion_Rate"].std(),
        df["Acquisition_Cost"].std(),
        df[["Clicks", "Impressions"]].corr().iloc[0, 1],
    ],
    "realistic": [
        realistic_df["ROI"].std(),
        realistic_df["ROI"].mean(),
        realistic_df["Conversion_Rate"].std(),
        realistic_df["Acquisition_Cost"].std(),
        realistic_df[["Clicks", "Impressions"]].corr(numeric_only=True).iloc[0, 1],
    ],
})

comparison

Saved realistic dataset to: C:\Users\hecto\OneDrive\Documentos\VSCode\py_marketing_eda\datasets\realistic_campaign_dataset.csv
Rows: 200000


,metric,cleaned,realistic
0,roi_std,1.734488,2.975303
1,roi_mean,5.002438,5.245692
2,conversion_rate_std,0.040602,0.047444
3,acquisition_cost_std,4337.664545,5847.306247
4,clicks_impressions_corr,0.000033,-0.177738
